# YOLOv5 Gun and Knife Detection Prototype

We will be using YOLOv5 as it is beginner friendly, it has fast training compared to the newer versions, it has numerous tutorials online and it can easily be custom trained.

## Importing Libraries

In [ ]:
import shutil
import os

import cv2
import torch
import numpy as np

from pathlib import Path
from google.colab import files

Enabling GPU to ensure faster training

In [ ]:
# make sure we are using GPU

# Check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available. Using", torch.cuda.get_device_name(0))
# If GPU is not available, use CPU
else:
    device = torch.device("cpu")
    print("GPU is not available. Using CPU.")

GPU is available. Using Tesla T4


## Mounting Google Drive to Colab for backing up purposes

In [ ]:
# mounting drive into this colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## YOLOv5 Setup

This part involves downloading the libraries and importing the YOLOv5 model.

In [ ]:
# installing ultralytics library to train YOLO model
!pip install --quiet ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.8 MB/s eta 0:00:00


In [ ]:
# cloning and installing yolov5 repo from github

!git clone https://github.com/ultralytics/yolov5 # clone
%cd yolov5
!pip install -r requirements.txt -q # install

Cloning into 'yolov5'...
remote: Enumerating objects: 17802, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 17802 (delta 7), reused 4 (delta 4), pack-reused 17788 (from 2)
Receiving objects: 100% (17802/17802), 17.00 MiB | 13.58 MiB/s, done.
Resolving deltas: 100% (12137/12137), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.5 MB/s eta 0:00:00


## Importing data

For this model, we will be using images of knives and guns from two different datasets from kaggle, as linked below:

1. https://www.kaggle.com/datasets/atulyakumar98/gundetection/data
2. https://www.kaggle.com/datasets/iqmansingh/guns-knives-object-detection?select=guns-knives-yolo


To view the pre-processing steps done, please visit the "ICT304_Assignment2_Pre-Processing.ipynb" Colab Notebook here: https://colab.research.google.com/drive/1Zlbu58mW3segMtkpe1ZnQ8vuiOmTftiv?usp=sharing

To run the code, please **download 'final_data.zip'** from the google drive here: https://drive.google.com/file/d/16tYWZVSqiwTlUnKWaMITSTqEY0OKWJXb/view?usp=drive_link

and then upload it to the run time in this Colab.

In [ ]:
# ensure we are in the right directory so that data folder can easily be unzipped below
%cd /content

/content


In [ ]:
# unzip data
!unzip -q final_data.zip -d data/

In [ ]:
# creat YAML file - stores configuration settings for YOLO in a human-readable format
data_yaml = """
train: /content/data/train/images
test: /content/data/test/images
val: /content/data/valid/images

nc: 2
names: ['knife', 'gun']
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

## Training YOLO Model

after importing the images and their labels in the YOLO format, and creating a YAML file, our model is ready to be trained. we are using 16 batch and 30 epochs for this trial run.

When running the code, a prompt will appear about wandb. Please type 3 in the output.

In [ ]:
# running YOLO model
%cd yolov5
!python train.py --img 640 --batch 16 --epochs 30 --data ../data.yaml --weights yolov5s.pt --patience 5

# automatically back up
# source folder containing all training runs
src_folder = '/content/yolov5/runs/train/'

# destination folder in gdrive
dst_folder = '/content/drive/MyDrive/yolo_results_backup/sixth_run'

# create destination if it doesn't exist
os.makedirs(dst_folder, exist_ok=True)

# copy entire training folder to Drive
shutil.copytree(src_folder, dst_folder, dirs_exist_ok=True)

Streaming output truncated to the last 5000 lines.
      20/29      4.41G    0.03736    0.01573   0.001906         30        640:  19% 107/558 [00:39<02:17,  3.27it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      20/29      4.41G    0.03734    0.01572   0.001899         37        640:  19% 108/558 [00:39<02:29,  3.00it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      20/29      4.41G    0.03742    0.01577    0.00189         52        640:  20% 109/558 [00:40<02:25,  3.09it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      20/29      4.41G    0.037

## Testing using a pre-recorded video

The following section tests the model by taking a pre-recorded video, identifying knives and guns inside it, and draws the bounding boxes onto the output. To run this section, please upload the pre-recorded video that can be found from our google drive, in this link: https://drive.google.com/file/d/1KrARSwa5MU6h1Publ82XioKMc1oOqXNg/view?usp=drive_link

In [ ]:
# import relevant libraries -- done here as these libraries don't exist before training model
from models.common import DetectMultiBackend
from utils.general import non_max_suppression, scale_boxes
from utils.torch_utils import select_device
from utils.augmentations import letterbox

In [ ]:
# Load trained model
weights_path = '/content/drive/MyDrive/yolo_results_backup/fourth_run/exp/weights/best.pt' # THIS PATH HAS TO BE EDITED BASED ON THE FOLDER NAME
device = select_device('0' if torch.cuda.is_available() else 'cpu')
model = DetectMultiBackend(weights_path, device=device)
stride, pt = model.stride, model.pt
imgsz = 640
model.eval()

# Input video path
video_path = '/content/ICT304_10s_knife_video.mp4'
cap = cv2.VideoCapture(video_path)

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Set up video writer
output_path = '/content/output_with_boxes.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Create output folder to save frames
output_folder = '/content/frames_with_boxes'
os.makedirs(output_folder, exist_ok=True)

# Process video frame-by-frame
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("End of video or failed to read frame.")
        break

    img0 = frame.copy()

    # Preprocess
    img = letterbox(img0, imgsz, stride=stride)[0]
    img = img.transpose((2, 0, 1))[::-1]  # HWC to CHW, BGR to RGB
    img = np.ascontiguousarray(img)

    img_tensor = torch.from_numpy(img).to(device)
    img_tensor = img_tensor.float() / 255.0
    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    # Inference
    with torch.no_grad():
        pred = model(img_tensor)
        pred = non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)

    # Draw predictions
    for det in pred:
        if len(det):
            det[:, :4] = scale_boxes(img_tensor.shape[2:], det[:, :4], img0.shape).round()
            for *xyxy, conf, cls in det:
                label = f'{model.names[int(cls)]} {conf:.2f}'
                cv2.rectangle(img0, (int(xyxy[0]), int(xyxy[1])),
                                   (int(xyxy[2]), int(xyxy[3])),
                                   (0, 255, 0), 2)
                cv2.putText(img0, label,
                            (int(xyxy[0]), int(xyxy[1]) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Write frame to video
    out.write(img0)
    frame_count += 1

    # Save frame as image
    frame_filename = os.path.join(output_folder, f"frame_{frame_count:04d}.jpg")
    cv2.imwrite(frame_filename, img0)
    frame_count += 1

# Finalize
cap.release()
out.release()
print(f"✅ Processed {frame_count} frames and saved to {output_path}")

# Auto-download in Colab
files.download(output_path)
